In [ ]:

import pandas as pd

train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/ml_benchmark/06_santander_customer/split_train.csv'
eval_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/ml_benchmark/06_santander_customer/split_eval.csv'

train_data = pd.read_csv(train_path)
eval_data = pd.read_csv(eval_path)

# Display the first few rows of the training data
print(train_data.head())


   target    var_0   var_1    var_2  ...   var_14   var_15   var_16   var_17
0       0  12.6407  2.9411  15.2365  ...  11.1089  15.0745   4.8570  -3.3979
1       0   5.5300 -2.6867  10.0739  ...   2.8759  13.9574   6.2063   6.6106
2       0   9.3848  1.3839   9.4459  ...  10.4819  14.2421   3.8085 -10.8337
3       0  11.5179 -5.7696  12.4163  ...   6.4419  14.8648   8.5745  -0.7653
4       0  12.6522 -6.0295  12.0836  ...   8.3703  14.4378  11.4569   0.4568

[5 rows x 19 columns]


In [ ]:

import numpy as np

# Check for missing values
print("Missing values in train data:\n", train_data.isnull().sum())
print("Missing values in eval data:\n", eval_data.isnull().sum())

# Check the distribution of the target column
print("Target distribution in train data:\n", train_data['target'].value_counts(normalize=True))
print("Target distribution in eval data:\n", eval_data['target'].value_counts(normalize=True))

# Summarize the statistics for numerical features
print("Statistics for numerical features in train data:\n", train_data.describe())


Missing values in train data:
 target    0
var_0     0
var_1     0
var_2     0
var_3     0
var_4     0
var_5     0
var_6     0
var_7     0
var_8     0
var_9     0
var_10    0
var_11    0
var_12    0
var_13    0
var_14    0
var_15    0
var_16    0
var_17    0
dtype: int64
Missing values in eval data:
 target    0
var_0     0
var_1     0
var_2     0
var_3     0
var_4     0
var_5     0
var_6     0
var_7     0
var_8     0
var_9     0
var_10    0
var_11    0
var_12    0
var_13    0
var_14    0
var_15    0
var_16    0
var_17    0
dtype: int64
Target distribution in train data:
 target
0    0.899825
1    0.100175
Name: proportion, dtype: float64
Target distribution in eval data:
 target
0    0.89825
1    0.10175
Name: proportion, dtype: float64
Statistics for numerical features in train data:
               target          var_0  ...         var_16         var_17
count  160000.000000  160000.000000  ...  160000.000000  160000.000000
mean        0.100175      10.677401  ...       9.337185     

In [ ]:


from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Separate features and target for training data
X_train = train_data.drop('target', axis=1)
y_train = train_data['target']

# Separate features and target for evaluation data
X_eval = eval_data.drop('target', axis=1)
y_eval = eval_data['target']

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled = scaler.transform(X_eval)

# Convert back to DataFrames for convenience
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_eval_scaled = pd.DataFrame(X_eval_scaled, columns=X_eval.columns)

print("Scaled training data head:\n", X_train_scaled.head())


Scaled training data head:
       var_0     var_1     var_2  ...    var_15    var_16    var_17
0  0.646003  1.127083  1.714367  ...  1.217056 -1.751116  0.342869
1 -1.693697 -0.261292 -0.242032  ... -1.494532 -1.223731  1.833321
2 -0.425317  0.742922 -0.480017  ... -0.803466 -2.160930 -0.764460
3  0.276557 -1.021842  0.645635  ...  0.708041 -0.298102  0.734912
4  0.649787 -1.085959  0.519556  ... -0.328435  0.828507  0.916906

[5 rows x 18 columns]


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Initialize and train the Logistic Regression model
log_reg = LogisticRegression(max_iter=10000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predict probabilities on the evaluation data
y_eval_prob = log_reg.predict_proba(X_eval_scaled)[:, 1]

# Calculate AUC
auc = roc_auc_score(y_eval, y_eval_prob)

print("AUC on eval data:", auc)



AUC on eval data: 0.6484759746463058


In [ ]:


# Create interaction terms between the first five features
for i in range(5):
    for j in range(i + 1, 5):
        col_name = f'var_{i}_var_{j}_interaction'
        X_train_scaled[col_name] = X_train_scaled[f'var_{i}'] * X_train_scaled[f'var_{j}']
        X_eval_scaled[col_name] = X_eval_scaled[f'var_{i}'] * X_eval_scaled[f'var_{j}']

print("Scaled training data head with interaction terms:\n", X_train_scaled.head())




Scaled training data head with interaction terms:
       var_0     var_1  ...  var_2_var_4_interaction  var_3_var_4_interaction
0  0.646003  1.127083  ...                -2.601724                 1.702245
1 -1.693697 -0.261292  ...                -0.336604                 1.002079
2 -0.425317  0.742922  ...                -0.153752                -0.258425
3  0.276557 -1.021842  ...                 0.254030                 0.301972
4  0.649787 -1.085959  ...                -0.895040                 1.731850

[5 rows x 28 columns]


In [ ]:



# Initialize and train the Logistic Regression model again with interaction terms
log_reg = LogisticRegression(max_iter=10000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predict probabilities on the evaluation data
y_eval_prob = log_reg.predict_proba(X_eval_scaled)[:, 1]

# Calculate AUC
auc = roc_auc_score(y_eval, y_eval_prob)

print("AUC on eval data with interaction terms:", auc)




AUC on eval data with interaction terms: 0.6477477431888787
